# 🚀 CRAFT Phase 3: Evaluate, Compare & Deploy

This notebook runs the final evaluation pipeline:

| Step | What it does |
|------|-------------|
| **1** | Clone repo & install dependencies |
| **2** | Mount your uploaded `checkpoint-250` dataset |
| **3** | Evaluate the **original Phi-3-Mini** (baseline scores) |
| **4** | Merge LoRA adapter + convert to GGUF + quantize |
| **5** | Evaluate **your CRAFT model** (quantized GGUF scores) |
| **6** | Auto-compare baseline vs CRAFT with a clear table |
| **7** | Upload to Hugging Face (only if improvement passes the cutoff) |

### Kaggle Secrets Required
- `GITHUB_PAT` — your GitHub Personal Access Token (to clone private repo)
- `HF_TOKEN` — your Hugging Face write token (for uploading the final model)

---
## Step 1: Setup Environment

In [ ]:
# 1a. Clone the repository
from kaggle_secrets import UserSecretsClient
import os
user_secrets = UserSecretsClient()
git_token = user_secrets.get_secret("GITHUB_PAT")
try:
    hf_token = user_secrets.get_secret("HF_TOKEN")
    if hf_token:
        import os
        os.environ['HF_TOKEN'] = hf_token
        from huggingface_hub import login
        login(token=hf_token)
        print("✅ Hugging Face Token authenticated globally!")
except Exception:
    print("⚠️ HF_TOKEN not found in secrets. Downloads might be rate-limited.")
username = "VishalGastu30"
!git clone https://{git_token}@github.com/{username}/CRAFT-SLM-Reasoning.git
%cd CRAFT-SLM-Reasoning
!git pull
print("✅ Repository cloned and up to date.")


In [ ]:
# 1b. Install dependencies
!pip install -q transformers bitsandbytes accelerate datasets peft trl loguru pyyaml scipy numpy huggingface_hub sentence-transformers
print("✅ All dependencies installed.")

---
## Step 2: Mount Checkpoint-250 from Kaggle Dataset

In [ ]:
import os
import shutil

print("Scanning /kaggle/input for checkpoint-250...")
rl_checkpoint = None

for root, dirs, files in os.walk('/kaggle/input'):
    if 'checkpoint-250' in dirs:
        rl_checkpoint = os.path.join(root, 'checkpoint-250')
        break
    elif 'adapter_model.safetensors' in files and '250' in root:
        rl_checkpoint = root
        break

if rl_checkpoint:
    print(f"🎉 Found checkpoint at: {rl_checkpoint}")
    target_dir = "/kaggle/working/CRAFT-SLM-Reasoning/checkpoints/rl/final"
    if os.path.exists(target_dir):
        shutil.rmtree(target_dir)
    os.makedirs(os.path.dirname(target_dir), exist_ok=True)
    shutil.copytree(rl_checkpoint, target_dir)
    print(f"✅ Copied to {target_dir}")
    print(f"📁 Files: {os.listdir(target_dir)}")
else:
    print("❌ Could not find checkpoint-250 in /kaggle/input!")
    print("Debug — top-level input folders:", os.listdir('/kaggle/input'))

# Verify
assert os.path.exists("checkpoints/rl/final/adapter_model.safetensors"), \
    "FATAL: adapter_model.safetensors not found in checkpoints/rl/final. Check your dataset upload!"
print("\n✅ Checkpoint-250 verified and ready.")

---
## Step 3: Evaluate the Original Phi-3-Mini Baseline
This downloads the untrained base model and runs it on GSM8K + StrategyQA to get your **before** scores.

In [ ]:
!python -m src.phase3_deploy.evaluator \
    --model-type hf \
    --model-path microsoft/Phi-3-mini-4k-instruct \
    --dataset all \
    --samples 50 \
    --gpu \
    --output-json craft_output/baseline_results.json

print("\n✅ Baseline evaluation complete. Results saved to craft_output/baseline_results.json")

---
## Step 4: Merge LoRA + Convert to GGUF + Quantize
This single step:
1. Merges your trained LoRA weights into the base Phi-3 model
2. Clones & compiles `llama.cpp` automatically
3. Converts the merged model to GGUF format
4. Quantizes to Q4_K_M (~2.2GB)

In [ ]:
# Step 4: Full merge + GGUF conversion + quantization pipeline
!python -m src.phase3_deploy.quantizer \
    --base-model microsoft/Phi-3-mini-4k-instruct \
    --lora-adapter checkpoints/rl/final \
    --merged-output craft_output/merged \
    --gguf-f16 craft_output/craft_phi3_f16.gguf \
    --gguf-quantized craft_output/craft_phi3_Q4_K_M.gguf

import os
assert os.path.exists("craft_output/craft_phi3_Q4_K_M.gguf"), f"FATAL: Quantized GGUF not found at craft_output/craft_phi3_Q4_K_M.gguf!"
size_mb = os.path.getsize("craft_output/craft_phi3_Q4_K_M.gguf") / (1024*1024)
print(f"\n✅ Quantization complete! Final model: {{size_mb:.0f}} MB")

---
## Step 5: Evaluate the CRAFT Model
Runs the same GSM8K + StrategyQA benchmarks on your quantized CRAFT model to get the **after** scores.

In [ ]:
# !CMAKE_ARGS="-DLLAMA_CUDA=on" pip install -q llama-cpp-python --upgrade --force-reinstall --no-cache-dir
!python -m src.phase3_deploy.evaluator \
    --model-type hf \
    --model-path craft_output/craft_phi3_Q4_K_M.gguf \
    --dataset all \
    --samples 50 \
    --gpu \
    --output-json craft_output/craft_results.json
# Verify the file was created
import os
assert os.path.exists("craft_output/craft_results.json"), \
    "FATAL: craft_results.json was not created! Check evaluator output above for errors."
print("\n✅ CRAFT evaluation complete. Results saved to craft_output/craft_results.json")


---
## Step 6: 📊 Compare Baseline vs CRAFT — The Verdict
This cell loads both result files, builds a comparison table, calculates the improvement, and tells you whether the model is ready to ship.

In [ ]:
import json, os

# ═══════════════════════════════════════════════════════════════
#  CONFIGURATION: Set your minimum improvement threshold here
# ═══════════════════════════════════════════════════════════════
MINIMUM_IMPROVEMENT_PERCENT = 3.0   # Model must improve by at least this % to pass
# ═══════════════════════════════════════════════════════════════

# Sanity check both files exist before loading
for f_path in ["craft_output/baseline_results.json", "craft_output/craft_results.json"]:
    assert os.path.exists(f_path), f"FATAL: {f_path} not found! Re-run the earlier steps."

# Load results
with open("craft_output/baseline_results.json", "r") as f:
    baseline = json.load(f)
with open("craft_output/craft_results.json", "r") as f:
    craft = json.load(f)

# Extract scores
datasets = ["gsm8k", "strategyqa"]
rows = []
deltas = []

for ds in datasets:
    b_acc = baseline.get(ds, {}).get("summary", {}).get("accuracy", 0) * 100
    c_acc = craft.get(ds, {}).get("summary", {}).get("accuracy", 0) * 100
    b_fmt = baseline.get(ds, {}).get("summary", {}).get("format_compliance", 0) * 100
    c_fmt = craft.get(ds, {}).get("summary", {}).get("format_compliance", 0) * 100
    b_rew = baseline.get(ds, {}).get("summary", {}).get("avg_reward", 0)
    c_rew = craft.get(ds, {}).get("summary", {}).get("avg_reward", 0)
    delta = c_acc - b_acc
    deltas.append(delta)
    rows.append((ds, b_acc, c_acc, delta, b_fmt, c_fmt, b_rew, c_rew))

avg_delta = sum(deltas) / len(deltas)

# Print the comparison table
print("\n" + "=" * 75)
print("  CRAFT Phase 3 — Benchmark Comparison Report")
print("=" * 75)

print(f"\n{'Dataset':<14} {'Baseline':>10} {'CRAFT':>10} {'Delta':>10} {'Status':>10}")
print("-" * 60)
for ds, b_acc, c_acc, delta, b_fmt, c_fmt, b_rew, c_rew in rows:
    status = "✅ PASS" if delta >= MINIMUM_IMPROVEMENT_PERCENT else "❌ FAIL"
    print(f"{ds:<14} {b_acc:>9.1f}% {c_acc:>9.1f}% {delta:>+9.1f}% {status:>10}")

print("-" * 60)
print(f"{'AVERAGE':<14} {'':>10} {'':>10} {avg_delta:>+9.1f}%")

print(f"\n{'Dataset':<14} {'Base Fmt':>10} {'CRAFT Fmt':>10} {'Base Rew':>10} {'CRAFT Rew':>10}")
print("-" * 60)
for ds, b_acc, c_acc, delta, b_fmt, c_fmt, b_rew, c_rew in rows:
    print(f"{ds:<14} {b_fmt:>9.1f}% {c_fmt:>9.1f}% {b_rew:>10.3f} {c_rew:>10.3f}")

# Final verdict
print("\n" + "=" * 75)
if avg_delta >= MINIMUM_IMPROVEMENT_PERCENT:
    print(f"  🏆 VERDICT: PASSED! Average improvement is +{avg_delta:.1f}% (threshold: +{MINIMUM_IMPROVEMENT_PERCENT}%)")
    print(f"  ✅ Model is ready for deployment. You can run Step 7 to upload.")
else:
    print(f"  ⚠️  VERDICT: DID NOT PASS. Average improvement is +{avg_delta:.1f}% (threshold: +{MINIMUM_IMPROVEMENT_PERCENT}%)")
    print(f"  🔄 Consider retraining with a capped curriculum, or use an earlier checkpoint.")
print("=" * 75)

# Save comparison for records
comparison = {
    "threshold_percent": MINIMUM_IMPROVEMENT_PERCENT,
    "passed": avg_delta >= MINIMUM_IMPROVEMENT_PERCENT,
    "average_improvement_percent": round(avg_delta, 2),
    "per_dataset": {ds: {"baseline": round(b, 2), "craft": round(c, 2), "delta": round(d, 2)} for ds, b, c, d, *_ in rows}
}
with open("craft_output/comparison.json", "w") as f:
    json.dump(comparison, f, indent=2)
print(f"\n📄 Full comparison saved to craft_output/comparison.json")

---
## Step 7: Upload to Hugging Face (Gated)
This cell will **only upload** if Step 6 passed the improvement threshold.  
Make sure your `HF_TOKEN` is set in Kaggle Secrets.

In [ ]:
import json
import os
from kaggle_secrets import UserSecretsClient

# Load the comparison verdict
with open("craft_output/comparison.json", "r") as f:
    verdict = json.load(f)

if not verdict["passed"]:
    print(f"🛑 Upload blocked. Model improvement was +{verdict['average_improvement_percent']}%")
    print(f"   Required: +{verdict['threshold_percent']}%")
    print(f"   The model did not meet the quality bar. Do NOT upload.")
else:
    print(f"✅ Model passed with +{verdict['average_improvement_percent']}% improvement.")
    
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    
    if hf_token:
        from huggingface_hub import HfApi
        api = HfApi(token=hf_token)
        
        repo_id = "Aurvion/CRAFT-Phi3-Mini"
        api.create_repo(repo_id=repo_id, repo_type="model", private=False, exist_ok=True)
        
        gguf_path = "craft_output/craft_phi3_Q4_K_M.gguf"
        assert os.path.exists(gguf_path), f"FATAL: {{gguf_path}} not found!"
        
        # Upload the quantized GGUF model
        api.upload_file(
            path_or_fileobj=gguf_path,
            path_in_repo="craft_phi3_Q4_K_M.gguf",
            repo_id=repo_id
        )
        
        # Upload the comparison report
        api.upload_file(
            path_or_fileobj="craft_output/comparison.json",
            path_in_repo="benchmark_comparison.json",
            repo_id=repo_id
        )
        
        print(f"\n🚀 Model uploaded to https://huggingface.co/{{repo_id}}")
        print(f"📄 Benchmark comparison also uploaded.")
    else:
        print("\n⚠️  HF_TOKEN not found in Kaggle Secrets.")
        print("   Add it via: Kaggle sidebar → Add-ons → Secrets → Add Secret")
        print("   Name: HF_TOKEN | Value: your Hugging Face write token")